In [ ]:
#%% PACKAGES
# Basics
import numpy as np                                                          
from math import pi                                                            
import warnings                                                              
import os    

from datetime import datetime, timedelta
from pandas import DataFrame
import geopandas as gpd

from sklearn.preprocessing import MinMaxScaler

# Visualization
import pandas as pd                                                          
import matplotlib.pyplot as plt                                                
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import numpy as np
import seaborn as sns; sns.set_theme(style='white')
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

#Read the files
IndexRaw = gpd.read_parquet(f'{output_step2_path}/step2_features.parquet')

attributs_info = pd.read_excel(f"{input_file_path}/attributs_info.xlsx", sheet_name="attributs_info")

In [ ]:
attributs_info

In [ ]:
# Fill NULL
Indexv1 = IndexRaw.fillna(0)

# Create mapping for processed columns while keeping first 5 columns unchanged
attribute_mapping = {}
for _, row in attributs_info.iterrows():
    old_name = f"{row['attribute']}_{row['method']}_{row['buffer_size']}"
    new_name = row['attribute']
    attribute_mapping[old_name] = new_name

# Keep first 5 columns as is, rename the rest using the mapping
first_5_cols = IndexRaw.columns[:5].tolist()
Indexv1 = Indexv1.rename(columns=attribute_mapping)

# Debug info
print("First 5 columns:", first_5_cols)
print("Renamed columns:", [col for col in Indexv1.columns if col not in first_5_cols])

Indexv1

In [ ]:
print('Crop outliers if necessary: --> see Walkability Amsterdam notebook')
Indexv2 = Indexv1.copy()
#Adding intervals (for factors where we have an interval of interest and below or above that interval the situation doesn't affect the walkability)
'''
example
Indexv1['amenities'] = Indexv1['amenities'].clip(upper=50) #more than 50 amenities in 350-meters-radius around segment centroid
'''

#Remove segments of less than 1m length
'''print("Keep only segment > 1 meter")
length_min = 1
Indexv2 = Indexv1[Indexv1['length'] > length_min]'''


In [ ]:
attributs_info

In [ ]:
# Create copy and normalize
Indexv3 = Indexv2.copy()
scaler = MinMaxScaler()

# Track changes during normalization
print("Min max normalization...")
for attribute in attributs_info['attribute']:
    if attribute in Indexv3.columns:
        # Normalize
        Indexv3.loc[:, attribute] = scaler.fit_transform(Indexv3[[attribute]]).round(4)
    else:
        print(f"Attribute '{attribute}' not found in Indexv3 columns.")

In [ ]:
# Inverse columns (for factors that have a negative effect on walkability) depending on attributs_info.impact_attribut (favorable or defavorable)
print("Inverse columns where necessary...")
for _, row in attributs_info.iterrows():
    attribute_name = row['attribute']
    impact = row['impact_attribut']
    if attribute_name in Indexv3.columns:
        if impact == 'defavorable':
            Indexv3[attribute_name] = 1 - Indexv3[attribute_name]
            print(f"Inverted attribute: {attribute_name}")
    else:
        print(f"Attribute '{attribute_name}' not found in Indexv3 columns.")


In [ ]:
Indexv3

In [ ]:
#Check data distribution (use this to check if there are still outliers skewing the factors)

Indexv3[['accident', 'rez_actif']].plot(kind='box', subplots=True, layout=(1, 4), figsize=(10, 4))
plt.tight_layout()
plt.show()


In [ ]:
# Calculate the Main Index Scores
Indexv4 = Indexv3

# Get weights from attributs_info and check validity
Zscore_weights = {}
for _, row in attributs_info[attributs_info.include_in_index == True].iterrows():
    attr = row['attribute']
    weight = row['initial_weight']
    if pd.isnull(weight):
        raise ValueError(f"Weight not specified for attribute '{attr}' (found NaN). Please specify a value between 0 and 1.")
    if not (0 <= weight <= 1):
        raise ValueError(f"Weight for attribute '{attr}' is {weight}, but must be between 0 and 1.")
    Zscore_weights[attr] = weight

print("Zscore_weights:", Zscore_weights)


# add zscore weights to the attributes_info dataframe
attributs_info['weights'] = attributs_info['attribute'].map(Zscore_weights)

# Function Sum-product to calculate sub-index score
def calculate_index(df, weights_dict):
    return sum(df[col] * weight for col, weight in weights_dict.items())

Indexv4['I-Zscore'] = calculate_index(Indexv4, Zscore_weights)
Indexv4['I-Zscore_unweighted'] = calculate_index(Indexv4, {key: 1 for key in Zscore_weights.keys()})

#Normalising Index
#Indexv4['Non-Scaled']=Indexv4['I-Zscore'].round(4)
Indexv4[['I-Zscore']] = scaler.fit_transform(Indexv4[['I-Zscore']]).round(4)
Indexv4[['I-Zscore_unweighted']] = scaler.fit_transform(Indexv4[['I-Zscore_unweighted']]).round(4)



In [ ]:

#%% CLASSES
# Définir les poids pour chaque classe
'''
print('Modify weights for sub-indexes ->>')
feasibility_weights = {
    'N-Furniture': 0.056,
    'N-Green': 0.048,
    'N-ParksPlaza': 0.056,
    'N-ParkinPres': 0.059,
    'N-ProxAmenities': 0.053,
}

accessibility_weights = {
    'N-RoadSafety': 0.094,
    'N-MaxSpeed': 0.073,
}

safety_weights = {
    'N-ProxAmenit': 0.073,
    'N-ShortBlocks': 0.031,
    'N-OV': 0.068,
}

comfort_weights = {
    'N-ProxAmenities': 0.059,
    'N-Crime': 0.071,
    'N-Lighting': 0.074,
}

pleasurability_weights = {
    'N-Obstacles': 0.093,
    'N-SidewalkWi': 0.086,
    'N-Maintenance': 0.065,
}


# Apply to Indexv4 and store results in Indexv5
Indexv5 = Indexv4.copy()
Indexv5['Feasibility'] = calculate_index(Indexv4, feasibility_weights)
Indexv5['Accessibility'] = calculate_index(Indexv4, accessibility_weights)
Indexv5['Safety'] = calculate_index(Indexv4, safety_weights)
Indexv5['Comfort'] = calculate_index(Indexv4, comfort_weights)
Indexv5['Pleasurability'] = calculate_index(Indexv4, pleasurability_weights)


# Min Max Normalisation
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
Indexv5[['Feasability','Accessibility','Safety','Comfort','Pleasurability']] = scaler.fit_transform(Indexv5[['Feasability','Accessibility','Safety','Comfort','Pleasurability']])

'''

In [ ]:
# Save it
# Convertir maxspeed en numérique (force les erreurs à NaN)
Indexv4['maxspeed'] = pd.to_numeric(Indexv4['maxspeed'], errors='coerce')


Indexv4.to_crs(target_crs).to_csv(f'{output_step3_path}/step3_index.csv', index = False)
Indexv4.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_index.parquet')

#After saving, perform a join-by-field value in QGIS with the shapefiles of the street segments. Join using the fields called "ID"